<a href="https://colab.research.google.com/github/yuki-gu/music-colab-notebooks/blob/main/MusicXML%E7%A7%BB%E8%AA%BF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MusicXMLの自動移調スクリプト

1. 移調設定で移調先の調を選択する
2. メニューバー → ランタイム → すべてのセルを実行
3. MusicXMLファイルのアップロード
    - ファイルを選択 → MusicXMLを選択 (複数可)
4. 移調後のMusicXMLを含むZipが自動的にダウンロードされる

In [ ]:
# @title 1. 移調設定 (-6 ~ +6)
minus_6 = True # @param {"type":"boolean"}
minus_5 = True # @param {"type":"boolean"}
minus_4 = True # @param {"type":"boolean"}
minus_3 = True # @param {"type":"boolean"}
minus_2 = True # @param {"type":"boolean"}
minus_1 = True # @param {"type":"boolean"}
plus_0 = False # @param {"type":"boolean"}
plus_1 = True # @param {"type":"boolean"}
plus_2 = True # @param {"type":"boolean"}
plus_3 = True # @param {"type":"boolean"}
plus_4 = True # @param {"type":"boolean"}
plus_5 = True # @param {"type":"boolean"}
plus_6 = False # @param {"type":"boolean"}

transpose_map = {
    "-d5": (minus_6, "m6"),
    "-P4": (minus_5, "m5"),
    "-M3": (minus_4, "m4"),
    "-m3": (minus_3, "m3"),
    "-M2": (minus_2, "m2"),
    "-m2": (minus_1, "m1"),
    "P1": (plus_0, "p0"),
    "m2": (plus_1, "p1"),
    "M2": (plus_2, "p2"),
    "m3": (plus_3, "p3"),
    "M3": (plus_4, "p4"),
    "P4": (plus_5, "p5"),
    "d5": (plus_6, "p6"),
}

In [ ]:
# @title 3. MusicXMLファイルのアップロード (複数対応)
from google.colab import files

uploaded = files.upload()

uploaded_files = list(uploaded.keys())
if len(uploaded_files) == 0:
    input_file = None
    print("ファイルがアップロードされていません。")
else:
    print("移調対象ファイル:", uploaded_files)

In [ ]:
# @title 環境設定
!pip install music21

In [ ]:
# @title 設定
output_dirname = "output" # @param {"type":"string"}
normalize_stem = True # @param {"type":"boolean"}
from pathlib import Path
output_dir = Path(output_dirname)

In [ ]:
# @title 符尾正規化の処理定義
"""
概要:
    MusicXMLの符尾の向きを正規化するスクリプト。
    入力MusicXMLを読み込み、音符・和音・連桁グループの符頭位置から
    <stem>up</stem> または <stem>down</stem> を設定する。

外部ファイル:
    読み込み:
    - FileSettings.input_file_path で指定した MusicXML / .musicxml / .xml ファイル

    書き込み:
    - FileSettings.output_file_path で指定した MusicXML / .musicxml / .xml ファイル

構成:
    - FileSettings: 入出力ファイルと実行設定
    - StemRuleSettings: 符尾判定の設定
    - ClefState: 現在の譜表ごとの音部記号情報
    - NoteEvent: 同じ符尾を共有する単音または和音
    - def main: 全体の実行処理
    - def normalize_musicxml_stem: MusicXMLの符尾を正規化する
    - def collect_note_events: MusicXMLから音符イベントを集める
    - def apply_stem_rules: 符尾ルールを適用する
    - def decide_single_event_stem: 単音または和音の符尾方向を決める
    - def decide_beam_group_stem: 連桁グループの符尾方向を決める
    - def get_pitch_position: 符頭の第3線からの相対位置を求める
"""

from dataclasses import dataclass, field
from pathlib import Path
import xml.etree.ElementTree as ElementTree


@dataclass(frozen=True)
class StemRuleSettings:
    """符尾ルールの設定"""

    upper_voice_stem: str = "up"  # 複数声部の上声の符尾方向
    lower_voice_stem: str = "down"  # 複数声部の下声の符尾方向
    middle_or_above_stem: str = "down"  # 第3線上または上の単音の符尾方向
    below_middle_stem: str = "up"  # 第3線より下の単音の符尾方向
    target_beam_number: str = "1"  # 主連桁として扱うbeam番号


@dataclass
class ClefState:
    """譜表上の第3線を推定するための音部記号情報"""

    sign: str = "G"
    line: int = 2
    octave_change: int = 0


@dataclass
class NoteEvent:
    """同じ符尾を共有する単音または和音"""

    notes: list[ElementTree.Element]
    part_id: str
    measure_number: str
    staff: str
    voice: str
    start_time: int
    position_values: list[int]
    stem: str | None = None

    def has_stem_target(self) -> bool:
        """符尾を設定する対象か確認する

        Returns:
            bool: 符尾を設定する対象ならTrue
        """
        return bool(self.position_values)


STEP_TO_INDEX = {
    "C": 0,
    "D": 1,
    "E": 2,
    "F": 3,
    "G": 4,
    "A": 5,
    "B": 6,
}

CLEF_TO_PITCH = {
    "G": ("G", 4),
    "F": ("F", 3),
    "C": ("C", 4),
}


# === XML補助関数 ===

def get_namespace(root: ElementTree.Element) -> str:
    """XML名前空間を取得する

    Args:
        root (ElementTree.Element): MusicXMLのルート要素

    Returns:
        str: 名前空間。名前空間がない場合は空文字列
    """
    if root.tag.startswith("{"):
        return root.tag.split("}")[0][1:]

    return ""


def make_tag(namespace: str, tag_name: str) -> str:
    """名前空間付きタグ名を作る

    Args:
        namespace (str): XML名前空間
        tag_name (str): タグ名

    Returns:
        str: ElementTreeで使うタグ名
    """
    if namespace:
        return f"{{{namespace}}}{tag_name}"

    return tag_name


def get_child_text(
        element: ElementTree.Element,
        namespace: str,
        tag_name: str,
        default_value: str = ""
) -> str:
    """子要素のテキストを取得する

    Args:
        element (ElementTree.Element): 親要素
        namespace (str): XML名前空間
        tag_name (str): 子要素のタグ名
        default_value (str): 子要素がない場合の値

    Returns:
        str: 子要素のテキスト
    """
    child = element.find(make_tag(namespace, tag_name))

    if child is None:
        return default_value

    if child.text is None:
        return default_value

    return child.text


def find_child(
        element: ElementTree.Element,
        namespace: str,
        tag_name: str
) -> ElementTree.Element | None:
    """子要素を取得する

    Args:
        element (ElementTree.Element): 親要素
        namespace (str): XML名前空間
        tag_name (str): 子要素のタグ名

    Returns:
        ElementTree.Element | None: 子要素。存在しない場合はNone
    """
    return element.find(make_tag(namespace, tag_name))


# === 五線位置の計算 ===

def pitch_to_diatonic_index(step: str, octave: int) -> int:
    """音名とオクターブを五線上の段数に変換する

    Args:
        step (str): C, D, E, F, G, A, Bのいずれか
        octave (int): MusicXMLのオクターブ

    Returns:
        int: C0を0とした五線上の段数
    """
    assert step in STEP_TO_INDEX

    return octave * 7 + STEP_TO_INDEX[step]


def get_middle_line_index(clef: ClefState) -> int:
    """音部記号から第3線の五線上の段数を求める

    Args:
        clef (ClefState): 音部記号情報

    Returns:
        int: 第3線の五線上の段数
    """
    if clef.sign not in CLEF_TO_PITCH:
        clef = ClefState()

    base_step, base_octave = CLEF_TO_PITCH[clef.sign]
    base_octave += clef.octave_change

    clef_pitch_index = pitch_to_diatonic_index(base_step, base_octave)
    middle_line_index = clef_pitch_index + (2 * (3 - clef.line))

    return middle_line_index


def get_pitch_position(
        note: ElementTree.Element,
        namespace: str,
        clef: ClefState
) -> int | None:
    """音符の符頭位置を第3線からの相対位置として求める

    第3線上を0、第3線より上を正、第3線より下を負とする。

    Args:
        note (ElementTree.Element): MusicXMLのnote要素
        namespace (str): XML名前空間
        clef (ClefState): 対応する譜表の音部記号

    Returns:
        int | None: 第3線からの相対位置。休符などpitchがない場合はNone
    """
    pitch = find_child(note, namespace, "pitch")

    if pitch is None:
        return None

    step = get_child_text(pitch, namespace, "step")
    octave_text = get_child_text(pitch, namespace, "octave")

    if not step or not octave_text:
        return None

    note_index = pitch_to_diatonic_index(step, int(octave_text))
    middle_line_index = get_middle_line_index(clef)

    return note_index - middle_line_index


# === 符尾対象の判定 ===

def is_note_with_stem(
        note: ElementTree.Element,
        namespace: str
) -> bool:
    """音符が符尾を持つ種類か確認する

    Args:
        note (ElementTree.Element): MusicXMLのnote要素
        namespace (str): XML名前空間

    Returns:
        bool: 符尾を持つ音符ならTrue
    """
    if find_child(note, namespace, "rest") is not None:
        return False

    note_type = get_child_text(note, namespace, "type")

    if note_type in {"whole", "breve", "long", "maxima"}:
        return False

    return find_child(note, namespace, "pitch") is not None


def get_note_duration(
        note: ElementTree.Element,
        namespace: str
) -> int:
    """note要素のdurationを取得する

    Args:
        note (ElementTree.Element): MusicXMLのnote要素
        namespace (str): XML名前空間

    Returns:
        int: duration。存在しない場合は0
    """
    duration_text = get_child_text(note, namespace, "duration", "0")

    if not duration_text:
        return 0

    return int(duration_text)


# === MusicXMLの解析 ===

def read_clef_from_element(
        clef_element: ElementTree.Element,
        namespace: str,
        current_clef: ClefState
) -> ClefState:
    """clef要素から音部記号情報を読む

    Args:
        clef_element (ElementTree.Element): clef要素
        namespace (str): XML名前空間
        current_clef (ClefState): 既存の音部記号情報

    Returns:
        ClefState: 更新後の音部記号情報
    """
    sign = get_child_text(clef_element, namespace, "sign", current_clef.sign)
    line_text = get_child_text(clef_element, namespace, "line", str(current_clef.line))
    octave_text = get_child_text(
        clef_element,
        namespace,
        "clef-octave-change",
        str(current_clef.octave_change)
    )

    return ClefState(
        sign=sign,
        line=int(line_text),
        octave_change=int(octave_text),
    )


def update_clef_states(
        attributes: ElementTree.Element,
        namespace: str,
        clef_by_staff: dict[str, ClefState]
) -> None:
    """attributes要素から譜表ごとの音部記号を更新する

    Args:
        attributes (ElementTree.Element): attributes要素
        namespace (str): XML名前空間
        clef_by_staff (dict[str, ClefState]): 譜表番号ごとの音部記号
    """
    clef_elements = attributes.findall(make_tag(namespace, "clef"))

    for clef_element in clef_elements:
        staff_number = clef_element.attrib.get("number", "1")
        current_clef = clef_by_staff.get(staff_number, ClefState())
        clef_by_staff[staff_number] = read_clef_from_element(
            clef_element,
            namespace,
            current_clef
        )


def make_note_event(
        note: ElementTree.Element,
        namespace: str,
        clef_by_staff: dict[str, ClefState],
        part_id: str,
        measure_number: str,
        current_time: int
) -> NoteEvent:
    """note要素から音符イベントを作る

    Args:
        note (ElementTree.Element): note要素
        namespace (str): XML名前空間
        clef_by_staff (dict[str, ClefState]): 譜表番号ごとの音部記号
        part_id (str): パートID
        measure_number (str): 小節番号
        current_time (int): 小節内の開始時刻

    Returns:
        NoteEvent: 音符イベント
    """
    staff = get_child_text(note, namespace, "staff", "1")
    voice = get_child_text(note, namespace, "voice", "1")
    clef = clef_by_staff.get(staff, ClefState())

    position = None

    if is_note_with_stem(note, namespace):
        position = get_pitch_position(note, namespace, clef)

    position_values = []

    if position is not None:
        position_values.append(position)

    return NoteEvent(
        notes=[note],
        part_id=part_id,
        measure_number=measure_number,
        staff=staff,
        voice=voice,
        start_time=current_time,
        position_values=position_values,
    )


def add_chord_note_to_event(
        event: NoteEvent,
        note: ElementTree.Element,
        namespace: str,
        clef_by_staff: dict[str, ClefState]
) -> None:
    """和音の追加音を既存イベントに追加する

    Args:
        event (NoteEvent): 追加先の音符イベント
        note (ElementTree.Element): chord要素を持つnote
        namespace (str): XML名前空間
        clef_by_staff (dict[str, ClefState]): 譜表番号ごとの音部記号
    """
    event.notes.append(note)

    if not is_note_with_stem(note, namespace):
        return

    clef = clef_by_staff.get(event.staff, ClefState())
    position = get_pitch_position(note, namespace, clef)

    if position is not None:
        event.position_values.append(position)


def collect_note_events(
        root: ElementTree.Element,
        namespace: str
) -> list[NoteEvent]:
    """MusicXMLから音符イベントを集める

    Args:
        root (ElementTree.Element): MusicXMLのルート要素
        namespace (str): XML名前空間

    Returns:
        list[NoteEvent]: 音符イベント一覧
    """
    events = []
    part_elements = root.findall(make_tag(namespace, "part"))

    for part in part_elements:
        part_id = part.attrib.get("id", "")
        clef_by_staff = {"1": ClefState()}

        measure_elements = part.findall(make_tag(namespace, "measure"))

        for measure in measure_elements:
            current_time = 0
            last_event = None
            measure_number = measure.attrib.get("number", "")

            for child in list(measure):
                if child.tag == make_tag(namespace, "attributes"):
                    update_clef_states(child, namespace, clef_by_staff)
                    continue

                if child.tag == make_tag(namespace, "backup"):
                    current_time -= get_note_duration(child, namespace)
                    continue

                if child.tag == make_tag(namespace, "forward"):
                    current_time += get_note_duration(child, namespace)
                    continue

                if child.tag != make_tag(namespace, "note"):
                    continue

                is_chord_note = find_child(child, namespace, "chord") is not None

                if is_chord_note and last_event is not None:
                    add_chord_note_to_event(
                        last_event,
                        child,
                        namespace,
                        clef_by_staff
                    )
                    continue

                last_event = make_note_event(
                    child,
                    namespace,
                    clef_by_staff,
                    part_id,
                    measure_number,
                    current_time
                )
                events.append(last_event)

                current_time += get_note_duration(child, namespace)

    return events


# === 符尾ルール ===

def decide_single_event_stem(
        position_values: list[int],
        settings: StemRuleSettings
) -> str:
    """単音または和音の符尾方向を決める

    Args:
        position_values (list[int]): 第3線からの相対位置
        settings (StemRuleSettings): 符尾ルール設定

    Returns:
        str: "up" または "down"
    """
    assert position_values

    if len(position_values) == 1:
        if position_values[0] >= 0:
            return settings.middle_or_above_stem

        return settings.below_middle_stem

    top_position = max(position_values)
    bottom_position = min(position_values)
    top_distance = abs(top_position)
    bottom_distance = abs(bottom_position)

    if top_distance > bottom_distance:
        return "down"

    if bottom_distance > top_distance:
        return "up"

    if len(position_values) == 2:
        return "down"

    above_count = len([value for value in position_values if value >= 0])
    below_count = len(position_values) - above_count

    if above_count >= below_count:
        return "down"

    return "up"


def get_event_representative_position(event: NoteEvent) -> int:
    """連桁判定用に音符イベントの代表位置を求める

    Args:
        event (NoteEvent): 音符イベント

    Returns:
        int: 第3線から最も遠い符頭の位置
    """
    assert event.position_values

    sorted_values = sorted(
        event.position_values,
        key=lambda value: (abs(value), value),
        reverse=True
    )

    return sorted_values[0]


def decide_beam_group_stem(
        events: list[NoteEvent],
        settings: StemRuleSettings
) -> str:
    """連桁グループの符尾方向を決める

    Args:
        events (list[NoteEvent]): 連桁でつながった音符イベント
        settings (StemRuleSettings): 符尾ルール設定

    Returns:
        str: "up" または "down"
    """
    target_events = [event for event in events if event.has_stem_target()]
    assert target_events

    representative_values = [
        get_event_representative_position(event)
        for event in target_events
    ]

    if all(value >= 0 for value in representative_values):
        return "down"

    if all(value < 0 for value in representative_values):
        return "up"

    if len(representative_values) == 2:
        first_value = representative_values[0]
        second_value = representative_values[1]

        if abs(first_value) == abs(second_value):
            return "down"

        if abs(first_value) > abs(second_value):
            return "down" if first_value >= 0 else "up"

        return "down" if second_value >= 0 else "up"

    above_count = len([value for value in representative_values if value >= 0])
    below_count = len(representative_values) - above_count

    if above_count > below_count:
        return "down"

    if below_count > above_count:
        return "up"

    max_distance = max(abs(value) for value in representative_values)
    farthest_values = [
        value
        for value in representative_values
        if abs(value) == max_distance
    ]

    if len({abs(value) for value in farthest_values}) == 1:
        has_above = any(value >= 0 for value in farthest_values)
        has_below = any(value < 0 for value in farthest_values)

        if has_above and has_below:
            return "down"

    farthest_value = farthest_values[0]

    return "down" if farthest_value >= 0 else "up"


def group_events_by_measure_staff(
        events: list[NoteEvent]
) -> dict[tuple[str, str, str], list[NoteEvent]]:
    """音符イベントをパート・小節・譜表ごとにまとめる

    Args:
        events (list[NoteEvent]): 音符イベント一覧

    Returns:
        dict[tuple[str, str, str], list[NoteEvent]]: グループ化した音符イベント
    """
    event_groups = {}

    for event in events:
        key = (event.part_id, event.measure_number, event.staff)
        event_groups.setdefault(key, []).append(event)

    return event_groups


def apply_multi_voice_rule(
        events: list[NoteEvent],
        settings: StemRuleSettings
) -> None:
    """一つの五線に複数声部がある場合の符尾方向を設定する

    Args:
        events (list[NoteEvent]): 音符イベント一覧
        settings (StemRuleSettings): 符尾ルール設定
    """
    event_groups = group_events_by_measure_staff(events)

    for group_events in event_groups.values():
        voice_to_events = {}

        for event in group_events:
            if event.has_stem_target():
                voice_to_events.setdefault(event.voice, []).append(event)

        if len(voice_to_events) < 2:
            continue

        voice_average_list = []

        for voice, voice_events in voice_to_events.items():
            values = [
                value
                for event in voice_events
                for value in event.position_values
            ]
            average_position = sum(values) / len(values)
            voice_average_list.append((voice, average_position))

        voice_average_list.sort(key=lambda item: item[1], reverse=True)

        upper_voice = voice_average_list[0][0]
        lower_voice = voice_average_list[-1][0]

        for event in voice_to_events[upper_voice]:
            event.stem = settings.upper_voice_stem

        for event in voice_to_events[lower_voice]:
            event.stem = settings.lower_voice_stem


def get_primary_beam_text(
        event: NoteEvent,
        namespace: str,
        settings: StemRuleSettings
) -> str:
    """音符イベントの主連桁状態を取得する

    Args:
        event (NoteEvent): 音符イベント
        namespace (str): XML名前空間
        settings (StemRuleSettings): 符尾ルール設定

    Returns:
        str: begin, continue, endなど。ない場合は空文字列
    """
    first_note = event.notes[0]
    beam_elements = first_note.findall(make_tag(namespace, "beam"))

    for beam in beam_elements:
        beam_number = beam.attrib.get("number", "1")

        if beam_number == settings.target_beam_number:
            return beam.text or ""

    return ""


def apply_beam_rule(
        events: list[NoteEvent],
        namespace: str,
        settings: StemRuleSettings
) -> None:
    """連桁グループの符尾方向を設定する

    Args:
        events (list[NoteEvent]): 音符イベント一覧
        namespace (str): XML名前空間
        settings (StemRuleSettings): 符尾ルール設定
    """
    event_groups = group_events_by_measure_staff(events)

    for group_events in event_groups.values():
        voice_to_events = {}

        for event in group_events:
            voice_to_events.setdefault(event.voice, []).append(event)

        for voice_events in voice_to_events.values():
            beam_group = []

            for event in voice_events:
                beam_text = get_primary_beam_text(event, namespace, settings)

                if beam_text == "begin":
                    beam_group = [event]
                    continue

                if beam_text == "continue" and beam_group:
                    beam_group.append(event)
                    continue

                if beam_text == "end" and beam_group:
                    beam_group.append(event)

                    if len(beam_group) >= 2:
                        stem = decide_beam_group_stem(beam_group, settings)

                        for beam_event in beam_group:
                            if beam_event.stem is None:
                                beam_event.stem = stem

                    beam_group = []


def apply_single_event_rule(
        events: list[NoteEvent],
        settings: StemRuleSettings
) -> None:
    """単音または和音の符尾方向を設定する

    Args:
        events (list[NoteEvent]): 音符イベント一覧
        settings (StemRuleSettings): 符尾ルール設定
    """
    for event in events:
        if event.stem is not None:
            continue

        if not event.has_stem_target():
            continue

        event.stem = decide_single_event_stem(event.position_values, settings)


def apply_stem_rules(
        events: list[NoteEvent],
        namespace: str,
        settings: StemRuleSettings
) -> None:
    """全符尾ルールを音符イベントに適用する

    Args:
        events (list[NoteEvent]): 音符イベント一覧
        namespace (str): XML名前空間
        settings (StemRuleSettings): 符尾ルール設定
    """
    apply_multi_voice_rule(events, settings)
    apply_beam_rule(events, namespace, settings)
    apply_single_event_rule(events, settings)


# === MusicXMLへの書き戻し ===

def find_stem_insert_index(
        note: ElementTree.Element,
        namespace: str
) -> int:
    """stem要素を挿入する位置を求める

    Args:
        note (ElementTree.Element): note要素
        namespace (str): XML名前空間

    Returns:
        int: stem要素の挿入位置
    """
    insert_before_tags = [
        "notehead",
        "beam",
        "notations",
        "lyric",
    ]

    target_tags = {
        make_tag(namespace, tag_name)
        for tag_name in insert_before_tags
    }

    for idx, child in enumerate(list(note)):
        if child.tag in target_tags:
            return idx

    return len(list(note))


def set_note_stem(
        note: ElementTree.Element,
        namespace: str,
        stem: str
) -> None:
    """note要素のstemを設定する

    Args:
        note (ElementTree.Element): note要素
        namespace (str): XML名前空間
        stem (str): "up" または "down"
    """
    assert stem in {"up", "down"}

    stem_element = find_child(note, namespace, "stem")

    if stem_element is None:
        stem_element = ElementTree.Element(make_tag(namespace, "stem"))
        insert_index = find_stem_insert_index(note, namespace)
        note.insert(insert_index, stem_element)

    stem_element.text = stem


def write_stems_to_xml(
        events: list[NoteEvent],
        namespace: str
) -> None:
    """音符イベントの符尾方向をMusicXMLへ書き戻す

    Args:
        events (list[NoteEvent]): 音符イベント一覧
        namespace (str): XML名前空間
    """
    for event in events:
        if event.stem is None:
            continue

        for note in event.notes:
            if is_note_with_stem(note, namespace):
                set_note_stem(note, namespace, event.stem)


def normalize_musicxml_stem(
        input_file_path: Path,
        output_file_path: Path,
        settings: StemRuleSettings
) -> None:
    """MusicXMLの符尾方向を正規化する

    Args:
        input_file_path (Path): 入力MusicXMLファイル
        output_file_path (Path): 出力MusicXMLファイル
        settings (StemRuleSettings): 符尾ルール設定

    Raises:
        FileNotFoundError: 入力ファイルが存在しない場合
    """
    if not input_file_path.exists():
        raise FileNotFoundError(f"入力ファイルが見つかりません: {input_file_path}")

    tree = ElementTree.parse(input_file_path)
    root = tree.getroot()
    namespace = get_namespace(root)

    if namespace:
        ElementTree.register_namespace("", namespace)

    events = collect_note_events(root, namespace)
    apply_stem_rules(events, namespace, settings)
    write_stems_to_xml(events, namespace)

    output_file_path.parent.mkdir(parents=True, exist_ok=True)
    tree.write(
        output_file_path,
        encoding="utf-8",
        xml_declaration=True
    )


# === テスト ===

def run_self_test() -> None:
    """符尾判定ロジックの簡易テストを行う"""
    settings = StemRuleSettings()

    assert decide_single_event_stem([0], settings) == "down"
    assert decide_single_event_stem([1], settings) == "down"
    assert decide_single_event_stem([-1], settings) == "up"

    assert decide_single_event_stem([4, -2], settings) == "down"
    assert decide_single_event_stem([2, -4], settings) == "up"
    assert decide_single_event_stem([2, -2], settings) == "down"

    assert decide_single_event_stem([4, 2, -4], settings) == "down"
    assert decide_single_event_stem([4, -2, -4], settings) == "up"

    first_event = NoteEvent(
        notes=[],
        part_id="P1",
        measure_number="1",
        staff="1",
        voice="1",
        start_time=0,
        position_values=[3],
    )
    second_event = NoteEvent(
        notes=[],
        part_id="P1",
        measure_number="1",
        staff="1",
        voice="1",
        start_time=1,
        position_values=[-2],
    )
    assert decide_beam_group_stem([first_event, second_event], settings) == "down"

    first_event.position_values = [2]
    second_event.position_values = [-2]
    assert decide_beam_group_stem([first_event, second_event], settings) == "down"

    third_event = NoteEvent(
        notes=[],
        part_id="P1",
        measure_number="1",
        staff="1",
        voice="1",
        start_time=2,
        position_values=[-1],
    )
    assert decide_beam_group_stem(
        [first_event, second_event, third_event],
        settings
    ) == "up"


def normalize_musicxml_stem_file(
        input_file_name: str,
        output_file_name: str
) -> None:
    """入力MusicXML名と出力MusicXML名を指定して符尾方向を正規化する

    Args:
        input_file_name (str): 入力MusicXMLファイル名
        output_file_name (str): 出力MusicXMLファイル名

    Raises:
        FileNotFoundError: 入力ファイルが存在しない場合
    """
    input_file_path = Path(input_file_name)
    output_file_path = Path(output_file_name)
    stem_rule_settings = StemRuleSettings()

    normalize_musicxml_stem(
        input_file_path,
        output_file_path,
        stem_rule_settings
    )

In [ ]:
# @title 移調処理
from pathlib import Path
from music21 import converter

for in_mxl in uploaded_files:
    in_mxl = Path(in_mxl)
    base = in_mxl.stem
    save_dir = output_dir / base
    save_dir.mkdir(parents=True, exist_ok=True)

    score = converter.parse(in_mxl)

    for interval_name, (is_checked, file_suffix) in transpose_map.items():
        if is_checked:
            transposed_score = score.transpose(interval_name)

            output_file = f"{base}_{file_suffix}.musicxml"
            output_path = save_dir / output_file

            transposed_score.write("musicxml", fp=output_path)

            if normalize_stem:
                normalize_musicxml_stem_file(output_path, output_path)

            print(f"作成: {output_path}")

In [ ]:
# @title 移調済みZipファイルのダウンロード
!zip -r "{output_dirname}.zip" "{output_dirname}"

from google.colab import files
files.download(f"{output_dirname}.zip")